In [157]:
print("reck98")
!(which python)
!(python --version)

reck98
/run/media/reck98/Others/Development/ML_NLP_DL/Ml-algorithms/.venv/bin/python
Python 3.12.12


In [158]:

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


from pathlib import Path as FilePath
# from rich import print
import string

import warnings
warnings.filterwarnings("ignore")

In [159]:
TEXT_FILE_PATH = FilePath("/run/media/reck98/Others/Development/ML_NLP_DL/Ml-algorithms/nlp/public/train.txt")

print(TEXT_FILE_PATH)


/run/media/reck98/Others/Development/ML_NLP_DL/Ml-algorithms/nlp/public/train.txt


In [160]:
original_df = pd.read_csv(TEXT_FILE_PATH, sep=";", header=None, names=["text", "emotion"])

original_df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [161]:
df = original_df.copy()

In [162]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [163]:
df['emotion'].value_counts()

emotion
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64

In [164]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text     16000 non-null  object
 1   emotion  16000 non-null  object
dtypes: object(2)
memory usage: 250.1+ KB


In [165]:
unique_emotions = df['emotion'].unique()

print(unique_emotions)

['sadness' 'anger' 'love' 'surprise' 'fear' 'joy']


In [166]:
emotions_numbers = {}

for emo in unique_emotions:
    emotions_numbers[emo] = unique_emotions.tolist().index(emo)
    
emotions_numbers


{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}

In [167]:
df['emotion'] = df['emotion'].map(emotions_numbers)
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [168]:
df['text'] = df['text'].apply(
    lambda x: x.lower()
)

In [169]:
def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))


df['text'] = df['text'].apply(remove_punctuation)
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [170]:
def remove_numbers(text = ""):
    new = ""
    
    for i in text:
        if not i.isdigit():
            new += i
    return new


df['text'] = df['text'].apply(remove_numbers)
df.head()


,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [171]:
def remove_emojis(text=""):
    new = ""

    for i in text:
        if i.isascii():
            new += i

    return new

df['text'] = df['text'].apply(remove_emojis)

In [172]:
texts = df['text'].tolist()

In [173]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /home/reck98/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/reck98/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [174]:
stopwords = set(stopwords.words('english'))
print(stopwords)

{"they've", 'i', "you'll", "they'd", 'same', "should've", 'up', "haven't", "needn't", 'such', 'off', 'because', 'into', 'ain', 'been', 'not', 'myself', 'this', 'him', 'am', 'have', "weren't", 'don', 'herself', 'through', 'yourself', 'above', 'isn', 'can', 'needn', 'will', 'me', 'any', 'should', 'we', 'all', 't', "you're", 'it', 'my', 'shouldn', 'so', 'ma', 'their', 'then', 'your', 'in', 'is', 'down', 'didn', 'had', 'her', 'll', "that'll", "they'll", 'are', 'than', 'nor', 'there', 'won', 'under', 'do', 'were', 'm', 'shan', 'd', 'his', "it'd", 'wasn', 'themselves', 'these', 'be', 'with', 'having', 'ours', 'own', 'and', 'a', 'after', 'himself', "wasn't", "couldn't", "shouldn't", 'aren', 'to', 'too', 'once', 'he', "he's", "we'd", 'does', 've', 'for', 'our', 'its', 'when', "you've", 'ourselves', "we're", 'both', 'most', 'here', "mightn't", 'again', 'until', 'during', 'doing', "i'll", 'more', 'wouldn', 'before', 'they', 'has', 'theirs', "isn't", "we'll", 'just', 'them', 'was', 'weren', 're',

In [175]:
len(stopwords)

198

In [176]:
i = 0

words = 0
for text in texts:
    
    newLine = ""
    
    for word in text.split(" "):
        words += 1
        if word not in stopwords:
            newLine += word + " "
            
    texts[i] = newLine
    i += 1

In [177]:
print(texts[0])

didnt feel humiliated 


In [178]:
df['text'] = texts
df.head()   

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [179]:
texts = df['text'].tolist()

In [180]:

'''
        Bag Of Words
'''

# from sklearn.feature_extraction.text import CountVectorizer

# documents = [
#     "Hello, how are you!",
#     "Win money, win from home.",
#     "The quick brown fox jumps over the lazy dog.",
#     "Hello, hello, hello.",
#     "Goodbye, goodbye, goodbye."
# ]

# vectorizer = CountVectorizer()

# X = vectorizer.fit_transform(documents)

# print(vectorizer.vocabulary_)
# print(X.toarray())

# print(len(X.toarray()[0]))

# {'hello': 6, 'how': 8, 'are': 0, 'you': 16, 'win': 15, 'money': 11, 'from': 4, 'home': 7, 'the': 14, 'quick': 13, 'brown': 1, 'fox': 3, 'jumps': 9, 'over': 12, 'lazy': 10, 'dog': 2, 'goodbye': 5}
# [[1 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1]
#  [0 0 0 0 1 0 0 1 0 0 0 1 0 0 0 2 0]
#  [0 1 1 1 0 0 0 0 0 1 1 0 1 1 2 0 0]
#  [0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0]
#  [0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0]]
# 17

'\n        Bag Of Words\n'

In [181]:
# '''
#         TF-IDF
# '''

# from sklearn.feature_extraction.text import TfidfVectorizer


# vectorizer = TfidfVectorizer()

# documents = [
#     "Hello, how are you!",
#     "Win money, win from home.",
#     "The quick brown fox jumps over the lazy dog.",
#     "Hello, hello, hello.",
#     "Goodbye, goodbye, goodbye."
# ]

# X = vectorizer.fit_transform(documents)

# print(vectorizer.vocabulary_)

# X = X.toarray()

# for row in X:
#     print(row)
#     print("")

In [182]:
X_train, X_test, y_train, y_test = train_test_split(df.drop("emotion", axis=1), df["emotion"], test_size=0.2, random_state=42)

In [183]:
bow_vectorizer = CountVectorizer()


# Learn vocabulary ONLY from training data
X_train_bow = bow_vectorizer.fit_transform(X_train['text'])

# Apply same vocabulary to test data
X_test_bow = bow_vectorizer.transform(X_test['text'])

print(X_train_bow.shape)

(12800, 13361)


In [184]:
naive_bayes_bow = MultinomialNB()

In [185]:
naive_bayes_bow.fit(X_train_bow, y_train)

y_pred_bow = naive_bayes_bow.predict(X_test_bow)

print(accuracy_score(y_test, y_pred_bow))

0.768125


In [186]:
naive_bayes_tfidf = MultinomialNB()

tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train['text'])

X_test_tfidf = tfidf_vectorizer.transform(X_test['text'])

naive_bayes_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = naive_bayes_tfidf.predict(X_test_tfidf)

print(accuracy_score(y_test, y_pred_tfidf))

0.6609375
